## Test Automation - sending logs to Application Insights

In this code sample we will explore how to send logs when building test automations for our LLM apps

In [13]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
APPLICATIONINSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

In [14]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    ContentSafetyEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
    ViolenceEvaluator,
    SexualEvaluator,
    SelfHarmEvaluator,
    HateUnfairnessEvaluator,
)
try:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    print(ex)
    

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
)

In [26]:
def test_coherence(test_id, query, response):
    coherence_evaluator = CoherenceEvaluator(model_config=model_config)
    score = coherence_evaluator(
        query=query, 
        response=response
    )
    score = 0 if score == None else score["gpt_coherence"]
    llm_properties = {
        'llm_version': AZURE_OPENAI_API_VERSION,
        'llm_model': AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
        'temperature': 0,
        'test_id': test_id,
        'test_type': 'coherence',
        'score': score
    }
    return llm_properties

def test_groundedness(test_id,response, context):
    groundedness_evaluator = GroundednessEvaluator(model_config=model_config)
    score = groundedness_evaluator(
        response=response,
        context=context,
    )
    score = 0 if score == None else score["gpt_groundedness"]
    llm_properties = {
        'llm_version': AZURE_OPENAI_API_VERSION,
        'llm_model': AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
        'temperature': 0,
        'test_id': test_id,
        'test_type': 'groundedness',
        'score': score
    }
    return llm_properties

def test_relevance(test_id, query, response, context):
    relevance_eval = RelevanceEvaluator(model_config=model_config)
    score = relevance_eval(
        query=query, 
        response=response,
        context=context,
    )
    score = 0 if score == None else score["gpt_relevance"]
    llm_properties = {
        'llm_version': AZURE_OPENAI_API_VERSION,
        'llm_model': AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
        'temperature': 0,
        'test_id': test_id,
        'test_type': 'relevance',
        'score': score
    }
    return llm_properties

In [16]:
import os
import logging

from opentelemetry._logs import set_logger_provider
from opentelemetry.sdk._logs import (
    LoggerProvider,
    LoggingHandler,
)
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from azure.monitor.opentelemetry.exporter import AzureMonitorLogExporter


def set_up_logging():
    logger_provider = LoggerProvider()
    set_logger_provider(logger_provider)

    exporter = AzureMonitorLogExporter(connection_string=APPLICATIONINSIGHTS_CONNECTION_STRING)
    logger_provider.add_log_record_processor(BatchLogRecordProcessor(exporter))

    # Attach LoggingHandler to namespaced logger
    handler = LoggingHandler()
    logger = logging.getLogger(__name__)
    logger.addHandler(handler)
    logger.setLevel(logging.NOTSET)
    return logger, logger_provider

In [17]:
# This must be done before any other telemetry calls
logger, logger_provider = set_up_logging()

In [31]:
query="What is the capital of France?"
response="Paris is the capital of France."
llm_properties = test_coherence(query, response)
logger.warning("Test automations", extra=llm_properties)

In [32]:
response="Paris is the capital of France."
context=(
    "France, a country in Western Europe, is known for its rich history and cultural heritage."
    "The city of Paris, located in the northern part of the country, serves as its capital."
    "Paris is renowned for its art, fashion, and landmarks such as the Eiffel Tower and the Louvre Museum."
)
llm_properties = test_groundedness(response, context)
logger.warning("Test automations", extra=llm_properties)

In [33]:
query="What is the capital of France?"
response="Paris is the capital of France."
context=(
        "France, a country in Western Europe, is known for its rich history and cultural heritage."
        "The city of Paris, located in the northern part of the country, serves as its capital."
        "Paris is renowned for its art, fashion, and landmarks such as the Eiffel Tower and the Louvre Museum."
        )
llm_properties = test_relevance(query, response, context)
logger.warning("Test automations", extra=llm_properties)

In [35]:
logger_provider.force_flush()

True

In [38]:
import pandas as pd
df = pd.read_csv("./data/test_data.csv")
for index, row in df.iterrows():
    test_id = row['test_id']
    test_type = row['test_type']
    query = row['query']
    answer = row['answer']
    context = row['context']
    if test_type == 'test_coherence':
        llm_properties = test_coherence(query, answer)
    elif test_type == 'test_groundedness':
        llm_properties = test_groundedness(answer, context)
    elif test_type == 'test_relevance':
        llm_properties = test_relevance(query, answer, context)
    logger.warning("Test automations", extra=llm_properties)

In [39]:
logger_provider.force_flush()

True